In [1]:
# Cell 1: Import libraries and check the remittance Excel file

import pandas as pd
import numpy as np
import os

# Check files in the current folder
os.listdir()

['BD_Election_2026_Next_Stage_Outputs',
 'Final_Diagnostics.ipynb',
 'Main_Datasheet_Rem.xlsx',
 'Model 1+2Linear Regression.ipynb',
 'Model_3_Remittance.ipynb.ipynb',
 'OLS_Model_1_Robust_Results.csv',
 'OLS_Model_2_With_Remittance_Robust_Results.csv',
 'OLS_Model_3_Margin_Remittance_Robust_Results.csv',
 'OLS_Model_3_Margin_Remittance_VIF.csv',
 'Table3_OLS_Combined.csv',
 'Table3_OLS_Combined.tex',
 'Untitled Folder',
 'sqlite.ipynb',
 'cpp-tiny-ray-tracer.ipynb',
 'Intro.ipynb',
 'Lorenz.ipynb',
 'cpp.ipynb',
 'r.ipynb',
 'cpp-third-party-libs.ipynb']

In [4]:
# Cell 2: Install openpyxl so pandas can read Excel files

try:
    import openpyxl
    print("openpyxl is already installed.")
except ModuleNotFoundError:
    try:
        import piplite
        await piplite.install("openpyxl")
    except ModuleNotFoundError:
        import micropip
        await micropip.install("openpyxl")

    import openpyxl
    print("openpyxl installed successfully:", openpyxl.__version__)

openpyxl installed successfully: 3.1.5


In [5]:
# Cell 2: Load the remittance Excel file and check sheet names

file_path = "Main_Datasheet_Rem.xlsx"

xls = pd.ExcelFile(file_path)

xls.sheet_names

['Constituency_Upazila_Map',
 'Population_data',
 'Crime_Data',
 'District_Crime_Unit_Map',
 'Election_Data',
 'Final_Merged_Data',
 'Constituency_Level_Data',
 'Master_Data',
 'Official_Constituencies',
 'Analysis_Data',
 'Remittance_Data',
 'Descriptive_Stats',
 'Correlation_Matrix',
 'Chart 1  Winning Party Seat Cou',
 'Chart 2 Average Winner Vote Sh',
 'Chart 3 Average Victory Margin',
 'Chat 4 Average Literacy Rate by',
 'Chat 5 Average Population Densi',
 'Chart 6 Average Total Crime Rat',
 'Chart 7 Average Violent Crime ',
 'Chart 8 Average Household Size ',
 'Chart 9 & 10 Metro_Comparison',
 'Chart 11 Literacy Rate vs Winn',
 'Chart 12 Density Vote Share',
 'Chart 13 Crime Vote Share',
 'Party_Comparison',
 'Chart 14 Remittance_EDA']

In [6]:
# Cell 3: Load the analysis sheet

df = pd.read_excel(file_path, sheet_name="Analysis_Data")

# Check the first few rows
df.head()

,Constituency,Districts,Upazilas,Upazila_Count,Area_sqkm,Population,Population_Density,Household_Size,Literacy_Rate,Crime_Units,...,Runner_Up_Votes,Runner_Up_Vote_Share,Margin_Votes,Margin_Percentage,Competitiveness,Metro,Use_For_Analysis,Remittance_HH_Total,CHECK,Remittance_HH_Urban
0,Bagerhat-1,Bagerhat,"Fakirhat, Mollahat, Chitalmari",3,813.53,456659,561.330252,4.072584,79.895851,Khulna Range,...,114323,49.309036,3204,1.381928,Highly Competitive,0,1,13393,9102,4291
1,Bagerhat-2,Bagerhat,"Bagerhat Sadar, Kachua",2,631.00,694132,1100.050713,4.053564,78.172080,Khulna Range,...,66409,36.068717,51300,27.862566,Safe Seat,0,1,13393,9102,4291
2,Bagerhat-3,Bagerhat,"Rampal, Mongla",2,1753.34,334508,190.783305,3.792103,81.073165,Khulna Range,...,83550,44.868456,19111,10.263089,Safe Seat,0,1,13393,9102,4291
3,Bagerhat-4,Bagerhat,"Morelganj, Sharankhola",2,624.80,425769,681.448464,3.801240,82.420112,Khulna Range,...,98326,45.862505,17741,8.274990,Safe Seat,0,1,13393,9102,4291
4,Bandarban-1,Bandarban,"Bandarban Sadar, Thanchi, Lama, Naikhongchhari...",7,4304.43,481093,111.766947,4.417808,63.630036,Chittagong Range,...,26162,15.608202,115293,68.783596,Safe Seat,0,1,4106,2100,2006


In [7]:
# Cell 4: Check all column names

df.columns.tolist()

['Constituency',
 'Districts',
 'Upazilas',
 'Upazila_Count',
 'Area_sqkm',
 'Population',
 'Population_Density',
 'Household_Size',
 'Literacy_Rate',
 'Crime_Units',
 'Total_Crime_per_100k',
 'Violent_Crime_per_100k',
 'Election_Status',
 'Winner',
 'Winning_Party',
 'Winning_Party_Code',
 'Winner_Votes',
 'Winner_Vote_Share',
 'Runner_Up',
 'Runner_Up_Votes',
 'Runner_Up_Vote_Share',
 'Margin_Votes',
 'Margin_Percentage',
 'Competitiveness',
 'Metro',
 'Use_For_Analysis',
 'Remittance_HH_Total',
 'CHECK',
 'Remittance_HH_Urban']

In [8]:
# Cell 5: Print column names clearly with numbers

for i, col in enumerate(df.columns):
    print(i, col)

0 Constituency
1 Districts
2 Upazilas
3 Upazila_Count
4 Area_sqkm
5 Population
6 Population_Density
7 Household_Size
8 Literacy_Rate
9 Crime_Units
10 Total_Crime_per_100k
11 Violent_Crime_per_100k
12 Election_Status
13 Winner
14 Winning_Party
15 Winning_Party_Code
16 Winner_Votes
17 Winner_Vote_Share
18 Runner_Up
19 Runner_Up_Votes
20 Runner_Up_Vote_Share
21 Margin_Votes
22 Margin_Percentage
23 Competitiveness
24 Metro
25 Use_For_Analysis
26 Remittance_HH_Total
27 CHECK
28 Remittance_HH_Urban


In [9]:
# Cell 6: Prepare clean data for Model 3

# Create log remittance column if it does not already exist
if "Log_Remittance_HH_Total" not in df.columns:
    df["Log_Remittance_HH_Total"] = np.log1p(pd.to_numeric(df["Remittance_HH_Total"], errors="coerce"))

# Variables for Model 3
model3_vars = [
    "Margin_Percentage",
    "Population_Density",
    "Household_Size",
    "Literacy_Rate",
    "Total_Crime_per_100k",
    "Violent_Crime_per_100k",
    "Metro",
    "Log_Remittance_HH_Total"
]

# Keep only analysis rows
df_model3 = df.copy()

df_model3["Use_For_Analysis"] = pd.to_numeric(df_model3["Use_For_Analysis"], errors="coerce")
df_model3 = df_model3[df_model3["Use_For_Analysis"] == 1]

# Convert model variables to numeric
for col in model3_vars:
    df_model3[col] = pd.to_numeric(df_model3[col], errors="coerce")

# Drop rows with missing values in model variables
df_model3 = df_model3[model3_vars].dropna()

print("Model 3 sample size:", len(df_model3))
df_model3.head()

Model 3 sample size: 297


,Margin_Percentage,Population_Density,Household_Size,Literacy_Rate,Total_Crime_per_100k,Violent_Crime_per_100k,Metro,Log_Remittance_HH_Total
0,1.381928,561.330252,4.072584,79.895851,102.321661,12.212238,0,9.502562
1,27.862566,1100.050713,4.053564,78.172080,102.321661,12.212238,0,9.502562
2,10.263089,190.783305,3.792103,81.073165,102.321661,12.212238,0,9.502562
3,8.274990,681.448464,3.801240,82.420112,102.321661,12.212238,0,9.502562
4,68.783596,111.766947,4.417808,63.630036,84.879645,10.309674,0,8.320448


In [11]:
# Cell 7: Run Model 3 OLS regression with HC3 robust standard errors

import statsmodels.api as sm

# Dependent variable
y = df_model3["Margin_Percentage"]

# Independent variables
X = df_model3[
    [
        "Population_Density",
        "Household_Size",
        "Literacy_Rate",
        "Total_Crime_per_100k",
        "Violent_Crime_per_100k",
        "Metro",
        "Log_Remittance_HH_Total"
    ]
]

# Add constant/intercept
X = sm.add_constant(X)

# Run OLS with HC3 robust standard errors
model3 = sm.OLS(y, X).fit(cov_type="HC3")

# Show full regression summary
print(model3.summary())

                            OLS Regression Results                            
Dep. Variable:      Margin_Percentage   R-squared:                       0.105
Model:                            OLS   Adj. R-squared:                  0.084
Method:                 Least Squares   F-statistic:                     6.036
Date:                Wed, 01 Jul 2026   Prob (F-statistic):           1.39e-06
Time:                        02:57:28   Log-Likelihood:                -1218.7
No. Observations:                 297   AIC:                             2453.
Df Residuals:                     289   BIC:                             2483.
Df Model:                           7                                         
Covariance Type:                  HC3                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [12]:
# Cell 8: Create clean Model 3 regression results table

# Confidence intervals
conf_int = model3.conf_int()
conf_int.columns = ["CI_Lower", "CI_Upper"]

# Build results table
results_table_3 = pd.DataFrame({
    "Variable": model3.params.index,
    "Coefficient": model3.params.values,
    "Robust_Std_Error": model3.bse.values,
    "P_Value": model3.pvalues.values,
    "CI_Lower": conf_int["CI_Lower"].values,
    "CI_Upper": conf_int["CI_Upper"].values
})

# Add significance stars
def significance_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    elif p < 0.1:
        return "."
    else:
        return ""

results_table_3["Significance"] = results_table_3["P_Value"].apply(significance_stars)

# Round values
results_table_3 = results_table_3.round({
    "Coefficient": 4,
    "Robust_Std_Error": 4,
    "P_Value": 4,
    "CI_Lower": 4,
    "CI_Upper": 4
})

results_table_3

,Variable,Coefficient,Robust_Std_Error,P_Value,CI_Lower,CI_Upper,Significance
0,const,37.2226,27.8926,0.1820,-17.4458,91.8911,
1,Population_Density,-0.0003,0.0002,0.0286,-0.0006,-0.0000,*
2,Household_Size,1.2428,2.6755,0.6423,-4.0010,6.4867,
3,Literacy_Rate,-0.1000,0.1691,0.5541,-0.4314,0.2314,
4,Total_Crime_per_100k,-0.3285,0.1178,0.0053,-0.5595,-0.0976,**
5,Violent_Crime_per_100k,0.4333,0.3672,0.2380,-0.2865,1.1530,
6,Metro,-2.8935,3.8454,0.4518,-10.4303,4.6432,
7,Log_Remittance_HH_Total,1.0942,1.2188,0.3693,-1.2946,3.4829,


In [13]:
# Cell 9: Check VIF for Model 3 predictors

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Use only independent variables, without dependent variable
X_vif = df_model3[
    [
        "Population_Density",
        "Household_Size",
        "Literacy_Rate",
        "Total_Crime_per_100k",
        "Violent_Crime_per_100k",
        "Metro",
        "Log_Remittance_HH_Total"
    ]
]

# Add constant
X_vif_const = sm.add_constant(X_vif)

# Calculate VIF
vif_table_3 = pd.DataFrame()
vif_table_3["Variable"] = X_vif_const.columns
vif_table_3["VIF"] = [
    variance_inflation_factor(X_vif_const.values, i)
    for i in range(X_vif_const.shape[1])
]

vif_table_3

,Variable,VIF
0,const,741.930219
1,Population_Density,2.513403
2,Household_Size,1.647589
3,Literacy_Rate,1.523914
4,Total_Crime_per_100k,2.601121
5,Violent_Crime_per_100k,1.754398
6,Metro,2.390961
7,Log_Remittance_HH_Total,1.870715


In [14]:
# Cell 10: Save Model 3 regression and VIF results

results_table_3.to_csv("OLS_Model_3_Margin_Remittance_Robust_Results.csv", index=False)
vif_table_3.to_csv("OLS_Model_3_Margin_Remittance_VIF.csv", index=False)

os.listdir()

['Main_Datasheet_Rem.xlsx',
 'Model 1+2Linear Regression.ipynb',
 'Model_3_Remittance.ipynb.ipynb',
 'OLS_Model_1_Robust_Results.csv',
 'OLS_Model_2_With_Remittance_Robust_Results.csv',
 'OLS_Model_3_Margin_Remittance_Robust_Results.csv',
 'OLS_Model_3_Margin_Remittance_VIF.csv',
 'Untitled Folder',
 'cpp-tiny-ray-tracer.ipynb',
 'cpp-third-party-libs.ipynb',
 'Lorenz.ipynb',
 'cpp.ipynb',
 'r.ipynb',
 'Intro.ipynb',
 'sqlite.ipynb']